In [128]:
from typing import Optional
from datetime import datetime
import pandas as pd
import numpy as np
import nfl_data_py as nfl

pd.set_option('display.max_columns', None)

# from data.schema.database import (
#     Player, NFLSeasonStats, NFLAdvancedStats,
#     NFLWeeklySnaps, InjuryRecord, get_session
# )

# Positions we care about for fantasy
SKILL_POSITIONS = {"QB", "RB", "WR", "TE"}

# Injury severity mapping — used for composite injury risk score
INJURY_SEVERITY = {
    "Knee": 0.8, "ACL": 1.0, "MCL": 0.7, "Meniscus": 0.7,
    "Hamstring": 0.6, "Quad": 0.5, "Groin": 0.5, "Hip": 0.5,
    "Ankle": 0.6, "Foot": 0.5,
    "Shoulder": 0.6, "Clavicle": 0.5, "Elbow": 0.4, "Wrist": 0.4,
    "Back": 0.7, "Ribs": 0.5,
    "Concussion": 0.8,
    "Illness": 0.2, "Rest": 0.0, "Not Injury Related": 0.0,
}

# Soft tissue injuries have higher recurrence risk
SOFT_TISSUE_INJURIES = {"Hamstring", "Quad", "Groin", "Hip", "Calf", "Thigh"}

In [148]:
def run_full_backfill(start_year: int = 2015, end_year: Optional[int] = None):
        """
        Master method — runs all ingestion steps in order.
        Pulls everything from start_year to present.
        Safe to re-run (upserts, not inserts).
        """
        if not end_year:
            end_year = datetime.now().year
        years = list(range(start_year, end_year + 1))

        print(f"Starting NFL backfill for {years[0]}–{years[-1]}")

        ingest_players()
        ingest_seasonal_stats(years)
        ingest_advanced_stats(years)
        ingest_snap_counts(years)
        ingest_injuries(years)

        print("NFL backfill complete.")

def _load_snap_pct_for_season_stats(years: list[int]) -> pd.DataFrame:
        """Aggregate snap % per player per season from weekly snap data."""
        try:
            snaps = nfl.import_snap_counts(years)
            snap_agg = (
                snaps[snaps["position"].isin(SKILL_POSITIONS)]
                .groupby(["pfr_player_id", "season"])
                .agg(snap_pct=("offense_pct", "mean"))
                .reset_index()
                .rename(columns={"pfr_player_id": "player_id"})
            )
            return snap_agg
        except Exception as e:
            print(f"Could not load snap data: {e}")
            return pd.DataFrame()
    
def _load_team_targets(years: list[int]) -> pd.DataFrame:
    """Compute team-level target totals (needed for target_share)."""
    try:
        pbp = nfl.import_pbp_data(years, columns=[
            "passer_player_id", "receiver_player_id", "pass_attempt",
            "posteam", "season", "play_type", "game_id"
        ])
        pass_plays = pbp[pbp["pass_attempt"] == 1]
        team_targets = (
            pass_plays.groupby(["posteam", "season"])
            .size()
            .reset_index(name="team_target_total")
        )
        return team_targets
    except Exception as e:
        print(f"Could not load team targets: {e}")
        return pd.DataFrame()
            
def ingest_seasonal_stats(years: list[int]):
        """
        Pulls seasonal aggregates: standard stats + fantasy points + efficiency.
        This is the core stats table.
        """
        print(f"Ingesting seasonal stats for {years}...")

        player_stats_df = nfl.import_seasonal_data(years, s_type="REG")
        # print("RAW DATA")
        # print(type(player_stats_df))
        # print(player_stats_df.columns)
        snap_df = _load_snap_pct_for_season_stats(years)
        # print("SNAP DATA")
        # print(type(snap_df))
        # print(snap_df.columns)
        team_targets = _load_team_targets(years)
        # print("TEAM TARGETS DATA")
        # print(type(team_targets))
        # print(team_targets.columns)
        
        # Merge snap pct and team targets into seasonal
        if not snap_df.empty:
            player_stats_df = player_stats_df.merge(snap_df, on=["player_id", "season"], how="left")
        if not team_targets.empty:
            player_stats_df = player_stats_df.merge(team_targets, on=["season"], how="left")

        # Compute derived metrics not in raw data
        player_stats_df["wopr"] = (1.5 * player_stats_df.get("target_share", 0)) + (0.7 * player_stats_df.get("air_yards_share", 0))
        player_stats_df["tgt_per_game"] = player_stats_df["targets"] / player_stats_df["games"].replace(0, float("nan"))
        player_stats_df["catch_rate"] = player_stats_df["receptions"] / player_stats_df["targets"].replace(0, float("nan"))
        player_stats_df["fantasy_ppg_ppr"] = player_stats_df["fantasy_points_ppr"] / player_stats_df["games"].replace(0, float("nan"))
        player_stats_df["fantasy_ppg_half"] = (player_stats_df.get("fantasy_points", player_stats_df["fantasy_points_ppr"] * 0.9)) / player_stats_df["games"].replace(0, float("nan"))
        player_stats_df["completion_pct"] = player_stats_df["completions"] / player_stats_df["attempts"].replace(0, float("nan"))

        records = []
        return player_stats_df
        # df = pd.DataFrame()
        # for _, row in player_stats_df.iterrows():
        #     # break
        #     print(row.columns)

        #     df_row = pd.DataFrame(
        #         [
        #             dict(
        #                 player_id=str(row["player_id"]),
        #                 season=int(row["season"]),
        #                 season_type="REG",
        #                 team=row.get("recent_team"),
        #                 games=_safe_int(row.get("games")),
        #                 games_started=_safe_int(row.get("games_started")),
        #                 completions=_safe_int(row.get("completions")),
        #                 attempts=_safe_int(row.get("attempts")),
        #                 passing_yards=_safe_int(row.get("passing_yards")),
        #                 passing_tds=_safe_int(row.get("passing_tds")),
        #                 interceptions=_safe_int(row.get("interceptions")),
        #                 passing_epa=row.get("passing_epa"),
        #                 completion_pct=row.get("completion_pct"),
        #                 yards_per_attempt=row.get("yards_per_attempt"),
        #                 passer_rating=row.get("passer_rating"),
        #                 sacks=_safe_int(row.get("sacks")),
        #                 carries=_safe_int(row.get("carries")),
        #                 rushing_yards=_safe_int(row.get("rushing_yards")),
        #                 rushing_tds=_safe_int(row.get("rushing_tds")),
        #                 rushing_epa=row.get("rushing_epa"),
        #                 yards_per_carry=row.get("rushing_yards_per_att"),
        #                 targets=_safe_int(row.get("targets")),
        #                 receptions=_safe_int(row.get("receptions")),
        #                 receiving_yards=_safe_int(row.get("receiving_yards")),
        #                 receiving_tds=_safe_int(row.get("receiving_tds")),
        #                 receiving_epa=row.get("receiving_epa"),
        #                 yards_per_reception=row.get("yards_per_reception"),
        #                 catch_rate=row.get("catch_rate"),
        #                 yards_per_target=row.get("yards_per_target"),
        #                 air_yards_total=_safe_int(row.get("receiving_air_yards")),
        #                 yards_after_catch=_safe_int(row.get("receiving_yards_after_catch")),
        #                 fantasy_points_ppr=row.get("fantasy_points_ppr"),
        #                 fantasy_ppg_ppr=row.get("fantasy_ppg_ppr"),
        #                 target_share=row.get("target_share"),
        #                 air_yards_share=row.get("air_yards_share"),
        #                 racr=row.get("racr"),
        #                 wopr=row.get("wopr"),
        #                 tgt_per_game=row.get("tgt_per_game"),
        #                 snap_pct=row.get("snap_pct")
        #             )
        #         ]
        #     )
            
        #     df = pd.concat([df, df_row], ignore_index=True)

        # self._bulk_upsert(NFLSeasonStats, records, conflict_column=("player_id", "season", "season_type"))
    
        print(f"{len(records)} seasonal stat rows ingested.")
        return df

def ingest_snap_counts(years: list[int]):
        """
        Weekly offensive snap counts. Useful for tracking role stability
        and detecting depth chart movement mid-season.
        """
        print(f"Ingesting weekly snap counts for {years}...")
        try:
            raw = nfl.import_snap_counts(years)
            raw = raw[raw["position"].isin(SKILL_POSITIONS)].copy()
        except Exception as e:
            print(f"Snap counts failed: {e}")
            return

        records = [
            NFLWeeklySnaps(
                player_id=str(row.get("pfr_player_id", "")),
                season=int(row["season"]),
                week=int(row["week"]),
                game_id=row.get("game_id", ""),
                team=row.get("team"),
                offense_snaps=_safe_int(row.get("offense_snaps")),
                offense_pct=row.get("offense_pct"),
                defense_snaps=_safe_int(row.get("defense_snaps")),
                st_snaps=_safe_int(row.get("st_snaps")),
            )
            for _, row in raw.iterrows()
        ]

        # self._bulk_upsert(NFLWeeklySnaps, records, conflict_column=None)  # no unique constraint here
        print(f"{len(records)} snap count rows ingested.")

In [197]:
# years = [2024]
years = list(range(2015, 2025)) #[2015, 2024]
# raw = ingest_seasonal_stats(years)
# raw = ingest_snap_counts(years)

player_stats_df = nfl.import_seasonal_data(years, s_type="REG")
snap_df = _load_snap_pct_for_season_stats(years)
team_targets = _load_team_targets(years)

2015 done.
2016 done.
2017 done.
2018 done.
2019 done.
2020 done.
2021 done.
2022 done.
2023 done.
2024 done.
Downcasting floats.


In [195]:
# Merge snap pct and team targets into seasonal
if not snap_df.empty:
    player_stats_df = player_stats_df.merge(snap_df, on=["player_id", "season"], how="left")
# if not team_targets.empty:
#     player_stats_df = player_stats_df.merge(team_targets, on=["season"], how="left")

# # Compute derived metrics not in raw data
player_stats_df["wopr"] = (1.5 * player_stats_df.get("target_share", 0)) + (0.7 * player_stats_df.get("air_yards_share", 0))

player_stats_df["tgt_per_game"] = player_stats_df["targets"] / player_stats_df["games"].replace(0, float("nan"))
player_stats_df["catch_rate"] = player_stats_df["receptions"] / player_stats_df["targets"].replace(0, float("nan"))
player_stats_df["fantasy_ppg_ppr"] = player_stats_df["fantasy_points_ppr"] / player_stats_df["games"].replace(0, float("nan"))
player_stats_df["fantasy_ppg_half"] = (player_stats_df.get("fantasy_points", player_stats_df["fantasy_points_ppr"] * 0.9)) / player_stats_df["games"].replace(0, float("nan"))
player_stats_df["completion_pct"] = player_stats_df["completions"] / player_stats_df["attempts"].replace(0, float("nan"))

2015 done.
2016 done.
2017 done.
2018 done.
2019 done.
2020 done.
2021 done.
2022 done.
2023 done.
2024 done.
Downcasting floats.


In [204]:
cols = player_stats_df.columns.tolist()
display(
    player_stats_df
    # .groupby('player_id')
    .groupby(["player_id", "season", "season_type"])
    .size()
    # .reset_index(name='Count')
    .sort_values(ascending=False)
)

player_id   season
00-0007091  2015      1
00-0034457  2019      1
00-0034487  2023      1
            2022      1
            2021      1
                     ..
00-0031512  2017      1
00-0031511  2019      1
            2017      1
            2015      1
00-0039921  2024      1
Length: 6098, dtype: int64

In [201]:
display(
    player_stats_df[player_stats_df['player_id'] == '00-0023459']
)

,player_id,season,season_type,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr_x,special_teams_tds,fantasy_points,fantasy_points_ppr,games,tgt_sh,ay_sh,yac_sh,wopr_y,ry_sh,rtd_sh,rfd_sh,rtdfd_sh,dom,w8dom,yptmpa,ppr_sh
98,00-0023459,2015,REG,347,572,3821.0,31,8.0,46.0,314.0,8,4,4629.0,1952.0,173.0,-7.692705,4,14.089957,1.525271,58,344.0,1,0.0,0.0,20.0,15.710952,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.000000,0,0.0,0.000000,0.000000,0.000000,0.0,301.24,301.24,16,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.213449
99,00-0023459,2016,REG,401,610,4428.0,40,7.0,35.0,246.0,5,2,5287.0,2030.0,222.0,132.419566,1,14.190815,2.590293,67,369.0,4,3.0,2.0,25.0,16.998949,1,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.000000,0,0.0,0.000000,0.000000,0.000000,0.0,380.02,380.02,16,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.232371
100,00-0023459,2017,REG,154,238,1675.0,16,6.0,22.0,168.0,1,1,1643.0,870.0,85.0,23.499506,0,6.736680,0.776788,24,126.0,0,0.0,0.0,10.0,4.662075,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.000000,0,0.0,0.000000,0.000000,0.000000,0.0,129.60,129.60,7,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.194454
101,00-0023459,2018,REG,372,597,4442.0,25,2.0,49.0,353.0,4,3,5241.0,2134.0,200.0,52.818098,2,13.611298,1.452469,43,269.0,2,2.0,0.0,20.0,21.600117,1,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.000000,0,0.0,0.000000,0.000000,0.000000,0.0,312.58,312.58,16,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.203402
102,00-0023459,2019,REG,353,569,4002.0,26,4.0,36.0,284.0,3,3,5006.0,2020.0,189.0,57.707451,2,13.523566,1.750914,46,183.0,1,1.0,1.0,9.0,-0.827063,1,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.000000,0,0.0,0.000000,0.000000,0.000000,0.0,278.38,278.38,16,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.191738
103,00-0023459,2020,REG,372,526,4299.0,48,5.0,20.0,182.0,1,1,4135.0,2248.0,216.0,186.251848,0,17.374026,3.555301,38,149.0,3,3.0,1.0,15.0,2.940303,0,1,1,-6.0,0,0.0,0.0,-4.0,-2.0,0.0,-1.355171,0,0.0,0.033333,-0.014440,0.039892,0.0,382.26,383.26,16,0.001901,-0.000967,-0.00089,0.002078,-0.001396,0.0,0.0,0.0,-0.000698,-0.001117,-0.011407,0.219299
104,00-0023459,2021,REG,366,531,4115.0,37,4.0,30.0,188.0,1,0,4084.0,2177.0,213.0,129.766062,0,16.838429,2.841165,33,101.0,3,2.0,0.0,10.0,5.831224,0,1,1,-4.0,0,0.0,0.0,-4.0,0.0,0.0,-1.243446,0,0.0,0.032258,-0.011299,0.040477,0.0,332.30,333.30,16,0.001789,-0.000929,0.00000,0.001940,-0.000923,0.0,0.0,0.0,-0.000461,-0.000738,-0.007156,0.207297
105,00-0023459,2022,REG,350,542,3695.0,26,12.0,32.0,258.0,4,2,4327.0,1893.0,177.0,-6.309946,2,16.405610,1.502478,34,94.0,1,2.0,2.0,9.0,-7.814974,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.000000,0,0.0,0.000000,0.000000,0.000000,0.0,239.20,239.20,17,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.166585
106,00-0023459,2023,REG,0,1,0.0,0,0.0,1.0,10.0,0,0,17.0,0.0,0.0,-2.031960,0,0.000000,0.000000,0,0.0,0,0.0,0.0,0.0,0.000000,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.000000,0,0.0,0.000000,0.000000,0.000000,0.0,0.00,0.00,1,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000
107,00-0023459,2024,REG,368,584,3897.0,28,11.0,40.0,302.0,5,2,4014.0,2119.0,192.0,10.130993,2,17.796653,1.367085,22,107.0,0,0.0,0.0,7.0,4.280564,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.000000,0,0.0,0.000000,0.000000,0.000000,0.0,256.58,256.58,17,0.000000,0.000000,0.00

In [141]:
display(
    pbp[pbp["passer_player_id"] == "00-0019596"]
)

,passer_player_id,receiver_player_id,pass_attempt,posteam,play_type,game_id,play_id,old_game_id,season,old_game_id_x,nflverse_game_id,old_game_id_y,possession_team,offense_formation,offense_personnel,defenders_in_box,defense_personnel,number_of_pass_rushers,players_on_play,offense_players,defense_players,n_offense,n_defense,ngs_air_yards,time_to_throw,was_pressure,route,defense_man_zone_type,defense_coverage_type,offense_names,defense_names,offense_positions,defense_positions,offense_numbers,defense_numbers
2302,00-0019596,00-0027150,1.0,NE,pass,2015_01_PIT_NE,305.0,2015091000,2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2303,00-0019596,00-0028087,1.0,NE,pass,2015_01_PIT_NE,346.0,2015091000,2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2304,00-0019596,00-0028087,1.0,NE,pass,2015_01_PIT_NE,371.0,2015091000,2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2305,00-0019596,00-0027656,1.0,NE,pass,2015_01_PIT_NE,396.0,2015091000,2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2306,00-0019596,00-0026035,1.0,NE,pass,2015_01_PIT_NE,418.0,2015091000,2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
382514,00-0019596,00-0038129,1.0,TB,pass,2022_19_DAL_TB,4221.0,NaN,2022,2023011600,2022_19_DAL_TB,2023011600,TB,SHOTGUN,"1 RB, 1 TE, 3 WR",5.0,"3 DL, 3 LB, 5 DB",3.0,44896;53504;53441;55075;39971;52421;25511;4608...,00-0033921;00-0037487;00-0036406;00-0019596;00...,00-0036942;00-0036932;00-0030575;00-0034674;00...,11.0,11.0,4.31,2.102,False,OUT,ZONE_COVERAGE,COVER_2,NaN,NaN,NaN,NaN,NaN,NaN
382515,00-0019596,00-0038129,1.0,TB,pass,2022_19_DAL_TB,4245.0,NaN,2022,2023011600,2022_19_DAL_TB,2023011600,TB,SHOTGUN,"1 RB, 1 TE, 3 WR",5.0,"3 DL, 3 LB, 5 DB",3.0,44896;53504;53441;55075;39971;52421;25511;4608...,00-0033921;00-0037487;00-0036406;00-0019596;00...,00-0036942;00-0036932;00-0030575;00-0034674;00...,11.0,11.0,17.48,2.469,False,POST,ZONE_COVERAGE,COVER_2,NaN,NaN,NaN,NaN,NaN,NaN
382516,00-0019596,None,1.0,TB,pass,2022_19_DAL_TB,4267.0,NaN,2022,2023011600,2022_19_DAL_TB,2023011600,TB,SHOTGUN,"1 RB, 1 TE, 3 WR",5.0,"3 DL, 3 LB, 5 DB",4.0,44896;53504;53441;39971;55075;52421;25511;4608...,00-0033921;00-0037487;00-0036406;00-0019596;00...,00-0036942;00-0036932;00-0030575;00-0034674;00...,11.0,11.0,NaN,NaN,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN
382518,00-0019596,00-0027944,1.0,TB,pass,2022_19_DAL_TB,4286.0,NaN,2022,2023011600,2022_19_DAL_TB,2023011600,TB,SHOTGUN,"1 RB, 1 TE, 3 WR",4.0,"2 DL, 4 LB, 5 DB",4.0,44896;53441;39971;52421;25511;54632;42377;5351...,00-0033921;00-0036406;00-0019596;00-0032217;00...,00-0036932;00-0030575;00-0037275;00-0036982;00...,11.0,11.0,3.10,3.037,False,CROSS,ZONE_COVERAGE,COVER_2,NaN,NaN,NaN,NaN,NaN,NaN


In [121]:
display(raw[raw['player_id']== '00-0034386'])

,player_id,season,season_type,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr_x,special_teams_tds,fantasy_points,fantasy_points_ppr,games,tgt_sh,ay_sh,yac_sh,wopr_y,ry_sh,rtd_sh,rfd_sh,rtdfd_sh,dom,w8dom,yptmpa,ppr_sh,snap_pct,wopr,tgt_per_game,catch_rate,fantasy_ppg_ppr,fantasy_ppg_half,completion_pct
130,00-0034386,2024,REG,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,0.0,0.0,0.0,0.0,0,22,32,289.0,2,0.0,0.0,486.0,68.0,13.0,19.086959,0,13.165941,1.024191,2.237225,3.102344,0.0,40.9,62.9,15,0.061069,0.143617,0.034154,0.206497,0.082454,0.08,0.067358,0.068807,0.081227,0.081963,0.551527,0.048377,NaN,3.102344,2.133333,0.6875,4.193333,2.726667,NaN


In [122]:
raw = nfl.import_snap_counts(years)
raw = raw[raw["position"].isin(SKILL_POSITIONS)].copy()

In [67]:
display(raw)

,game_id,pfr_game_id,season,game_type,week,player,pfr_player_id,position,team,opponent,offense_snaps,offense_pct,defense_snaps,defense_pct,st_snaps,st_pct
4,2015_01_BAL_DEN,201509130den,2015,REG,1,Peyton Manning,MannPe00,QB,DEN,BAL,70.0,1.00,0.0,0.0,0.0,0.00
6,2015_01_BAL_DEN,201509130den,2015,REG,1,Emmanuel Sanders,SandEm00,WR,DEN,BAL,65.0,0.93,0.0,0.0,6.0,0.21
7,2015_01_BAL_DEN,201509130den,2015,REG,1,Demaryius Thomas,ThomDe03,WR,DEN,BAL,61.0,0.87,0.0,0.0,0.0,0.00
8,2015_01_BAL_DEN,201509130den,2015,REG,1,Owen Daniels,DaniOw00,TE,DEN,BAL,61.0,0.87,0.0,0.0,0.0,0.00
9,2015_01_BAL_DEN,201509130den,2015,REG,1,C.J. Anderson,AndeC.00,RB,DEN,BAL,52.0,0.74,0.0,0.0,0.0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26585,2024_22_KC_PHI,202502090phi,2024,SB,22,Kareem Hunt,HuntKa00,RB,KC,PHI,11.0,0.20,0.0,0.0,0.0,0.00
26586,2024_22_KC_PHI,202502090phi,2024,SB,22,Justin Watson,WatsJu01,WR,KC,PHI,10.0,0.18,0.0,0.0,8.0,0.28
26608,2024_22_KC_PHI,202502090phi,2024,SB,22,Peyton Hendershot,HendPe01,TE,KC,PHI,0.0,0.00,0.0,0.0,19.0,0.66
26609,2024_22_KC_PHI,202502090phi,2024,SB,22,Nikko Remigio,RemiNi00,WR,KC,PHI,0.0,0.00,0.0,0.0,13.0,0.45
